In [6]:
import json

# 1. Definimos estrictamente los campos de tu CMD
CMD_FIELDS = [
    "id", 
    "created_at", 
    "age", 
    "specialty", 
    "text", 
    "code", 
    "url"
]

In [13]:
def limpiar_texto(texto):
    if not isinstance(texto, str):
        return texto
        
    # Diccionario de correcciones (puedes agregar más si descubres otros errores en tu JSON)
    correcciones = {
        "\uffe1": "á",
        "\uffe9": "é",
        "\uffed": "í",
        "\ufff3": "ó",
        "\ufffa": "ú",
        "\ufff1": "ñ",
        "\uffd1": "Ñ"
    }
    
    # Reemplazamos cada error por su caracter correcto
    texto_limpio = texto
    for error, correccion in correcciones.items():
        texto_limpio = texto_limpio.replace(error, correccion)
        
    # Opcional: Eliminar espacios extra al inicio y final
    return texto_limpio.strip()

In [16]:
def procesar_dataset_cmd(ruta_entrada, ruta_salida):
    # Abrir el JSON original
    with open(ruta_entrada, 'r', encoding='utf-8') as archivo_entrada:
        datos_crudos = json.load(archivo_entrada)
        
    dataset_filtrado = []

    # Iterar sobre cada registro del JSON
    for registro in datos_crudos.get("data", []):  # Asumiendo que los registros están bajo la clave "data"
        # 2. Filtrar solo los registros donde "correct" sea verdadero.
        # Manejamos el caso de que venga como Booleano (True) o como String ("true")
        estado_correcto = registro.get("correct")
        if estado_correcto is True or str(estado_correcto).lower() == "true":
            
            # 3. Construir el nuevo registro solo con los campos del CMD
            nuevo_registro = {}
            for campo in CMD_FIELDS:
                # Usamos .get() para evitar errores si un campo opcional (como edad o uri) no existe
                valor = registro.get(campo, None)
                if campo in ["text", "specialty", "url"] and valor is not None:
                    valor = limpiar_texto(valor)
                nuevo_registro[campo] = valor
                
            dataset_filtrado.append(nuevo_registro)

    # Guardar el nuevo JSON estructurado y limpio
    with open(ruta_salida, 'w', encoding='utf-8') as archivo_salida:
        json.dump(dataset_filtrado, archivo_salida, indent=4, ensure_ascii=False)

    # Resumen de consola
    print(f"Procesamiento exitoso.")
    print(f"Registros originales: {len(datos_crudos)}")
    print(f"Registros filtrados (correctos y estructurados): {len(dataset_filtrado)}")

In [17]:
if __name__ == "__main__":
    # Cambia estos nombres por las rutas reales de tus archivos
    archivo_input = "datos_cmd.json"
    archivo_output = "datos_cmd_limpio.json"
    
    procesar_dataset_cmd(archivo_input, archivo_output)

Procesamiento exitoso.
Registros originales: 5
Registros filtrados (correctos y estructurados): 198
